# BERT: five matched fine-tuning seeds

Run all cells in a **GPU runtime**. This study runs seeds `[17011, 29023, 41047, 53069, 65089]` independently from
the pinned original pretrained checkpoint, with the same fixed 65/50 policy split.
Extra MLM pretraining is disabled. Each complete seed uses five epochs (370 optimizer steps).

Final reports contain **mean Macro-F1 ± sample standard deviation (ddof=1)** for category,
hierarchical, and gated fine-grained scores. All five seeds are required; no result is pre-filled.
Use the same `STUDY_NAME` and seed list in all four notebooks. Each model writes to its own folder.
If Colab disconnects, rerun this notebook: verified completed seeds are reused, and an unfinished
seed restarts from its original pretrained checkpoint. This is seed-level, not mid-epoch, recovery.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, subprocess, sys, zipfile, re
from google.colab import drive, files
drive.mount('/content/drive')
MODEL = 'BERT'
ARCHIVE_NAME = 'BERT_multiseed_workspace.zip'
archive_path = Path('/content/drive/MyDrive/OPP115_MultiSeed/packages') / ARCHIVE_NAME
if not archive_path.is_file():
    print('Upload', ARCHIVE_NAME)
    uploaded = files.upload()
    if set(uploaded) != {ARCHIVE_NAME}:
        raise ValueError('Upload exactly ' + ARCHIVE_NAME)
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    archive_path.write_bytes(uploaded[ARCHIVE_NAME])
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
PROJECT_DIR = Path('/content') / (MODEL + '_multiseed_' + digest[:12])
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if len(archive.infolist()) > 5000 or sum(x.file_size for x in archive.infolist()) > 2 * 1024**3:
        raise ValueError('Archive exceeds workspace limits')
    for entry in archive.infolist():
        member = PurePosixPath(entry.filename)
        if member.is_absolute() or '..' in member.parts or (entry.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Unsafe archive member')
        destination = (PROJECT_DIR / entry.filename).resolve()
        destination.relative_to(PROJECT_DIR.resolve())
    archive.extractall(PROJECT_DIR)
manifest = json.loads((PROJECT_DIR / 'workspace_manifest.json').read_text())
if manifest['model'] != MODEL:
    raise ValueError('Wrong model archive')
for entry in manifest['files']:
    path = (PROJECT_DIR / entry['path']).resolve()
    path.relative_to(PROJECT_DIR.resolve())
    if path.stat().st_size != entry['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Archive integrity failure: ' + entry['path'])
print('Verified workspace:', PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements_colab_bert_opp115.txt')], check=True)


In [ ]:
MODE = 'full'  #@param ['full', 'smoke']
STUDY_NAME = 'matched_five_seeds_v1'  #@param {type:'string'}
SEEDS = [17011, 29023, 41047, 53069, 65089]
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]{0,79}', STUDY_NAME):
    raise ValueError('Use a simple study name with no path separators.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/OPP115_MultiSeed/studies') / STUDY_NAME
import torch
print('CUDA available:', torch.cuda.is_available())
if MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun.')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Mode:', MODE, 'Seeds:', SEEDS)
print('Results:', OUTPUT_ROOT / MODE / MODEL)


In [ ]:
command = [sys.executable, str(PROJECT_DIR / 'multiseed.py'), '--mode', MODE,
           '--output-root', str(OUTPUT_ROOT), '--seeds', *map(str, SEEDS)]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
from IPython.display import Markdown, display
report_path = OUTPUT_ROOT / MODE / MODEL / 'report.md'
display(Markdown(report_path.read_text()))
print('Per-seed scores:', report_path.parent / 'per_seed.csv')
print('Validated aggregate:', report_path.parent / 'completed_report.json')


## Combined four-model report
Run the following cell once all four notebooks have completed in full mode using the same study name.
It refuses partial studies, smoke runs, changed data, or mismatched seeds/settings.
The report includes paired PrivBERT-minus-LegalBERT differences, without declaring a winner from a single run.


In [ ]:
models = ['BERT', 'PrivBERT', 'RoBERTa', 'LegalBERT']
pending = [m for m in models if not (OUTPUT_ROOT / 'full' / m / 'completed_report.json').exists()]
if MODE != 'full':
    print('Smoke mode is not eligible for the research comparison.')
elif pending:
    print('Comparison pending. Complete these model notebooks:', ', '.join(pending))
else:
    subprocess.run([sys.executable, str(PROJECT_DIR / 'compare_models.py'),
                    '--output-root', str(OUTPUT_ROOT)], cwd=PROJECT_DIR, check=True)
    display(Markdown((OUTPUT_ROOT / 'full/comparison.md').read_text()))
